In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_1961/3320559344.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob5") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "8")\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 15:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
def build_host_interests(spark, logical_date, logger):
    """Скрипт сборки абонентской базы для spfix_feature_store"""

    import pyspark.sql.functions as F

    # report month for current build
    business_month = datetime(logical_date.year, logical_date.month, 1).date()
    business_month_end = str(business_month + relativedelta(months=1, days= -1))

    host_interests = (
        spark.table('gr_dm.feature_store_hosts_user_ratio_stats')
        .select(
            'regid', 'app_n', 'msisdn', 'category_id',
            'num_requests_to_overall_by_user',
            'num_active_days_to_overall_by_user',
            'num_requests_to_mean_by_category',
            'num_active_days_to_mean_by_category'
        )
    )

    for i in range(1, 96):
        host_interests = (
            host_interests
            .withColumn(f'cat_{i}_interest_num_requests_to_overall_by_user', F.when(
                F.col('category_id') == i, F.col('num_requests_to_overall_by_user')))
            .withColumn(f'cat_{i}_interest_num_active_days_to_overall_by_user', F.when(
                F.col('category_id') == i, F.col('num_active_days_to_overall_by_user')))
            .withColumn(f'cat_{i}_num_requests_to_mean_by_category', F.when(
                F.col('category_id') == i, F.col('num_requests_to_mean_by_category')))
            .withColumn(f'cat_{i}_num_active_days_to_mean_by_category', F.when(
                F.col('category_id') == i, F.col('num_active_days_to_mean_by_category')))
        )

    final_df = (
        host_interests
        .fillna(0)
        .groupBy('regid', 'app_n', 'msisdn')
        .agg(
            F.sum('cat_1_interest_num_requests_to_overall_by_user').alias(
                'cat_1_interest_num_requests_to_overall_by_user'),
            F.sum('cat_1_interest_num_active_days_to_overall_by_user').alias(
                'cat_1_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_1_num_requests_to_mean_by_category').alias('cat_1_num_requests_to_mean_by_category'),
            F.sum('cat_1_num_active_days_to_mean_by_category').alias('cat_1_num_active_days_to_mean_by_category'),
            F.sum('cat_2_interest_num_requests_to_overall_by_user').alias(
                'cat_2_interest_num_requests_to_overall_by_user'),
            F.sum('cat_2_interest_num_active_days_to_overall_by_user').alias(
                'cat_2_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_2_num_requests_to_mean_by_category').alias('cat_2_num_requests_to_mean_by_category'),
            F.sum('cat_2_num_active_days_to_mean_by_category').alias('cat_2_num_active_days_to_mean_by_category'),
            F.sum('cat_3_interest_num_requests_to_overall_by_user').alias(
                'cat_3_interest_num_requests_to_overall_by_user'),
            F.sum('cat_3_interest_num_active_days_to_overall_by_user').alias(
                'cat_3_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_3_num_requests_to_mean_by_category').alias('cat_3_num_requests_to_mean_by_category'),
            F.sum('cat_3_num_active_days_to_mean_by_category').alias('cat_3_num_active_days_to_mean_by_category'),
            F.sum('cat_4_interest_num_requests_to_overall_by_user').alias(
                'cat_4_interest_num_requests_to_overall_by_user'),
            F.sum('cat_4_interest_num_active_days_to_overall_by_user').alias(
                'cat_4_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_4_num_requests_to_mean_by_category').alias('cat_4_num_requests_to_mean_by_category'),
            F.sum('cat_4_num_active_days_to_mean_by_category').alias('cat_4_num_active_days_to_mean_by_category'),
            F.sum('cat_5_interest_num_requests_to_overall_by_user').alias(
                'cat_5_interest_num_requests_to_overall_by_user'),
            F.sum('cat_5_interest_num_active_days_to_overall_by_user').alias(
                'cat_5_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_5_num_requests_to_mean_by_category').alias('cat_5_num_requests_to_mean_by_category'),
            F.sum('cat_5_num_active_days_to_mean_by_category').alias('cat_5_num_active_days_to_mean_by_category'),
            F.sum('cat_6_interest_num_requests_to_overall_by_user').alias(
                'cat_6_interest_num_requests_to_overall_by_user'),
            F.sum('cat_6_interest_num_active_days_to_overall_by_user').alias(
                'cat_6_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_6_num_requests_to_mean_by_category').alias('cat_6_num_requests_to_mean_by_category'),
            F.sum('cat_6_num_active_days_to_mean_by_category').alias('cat_6_num_active_days_to_mean_by_category'),
            F.sum('cat_7_interest_num_requests_to_overall_by_user').alias(
                'cat_7_interest_num_requests_to_overall_by_user'),
            F.sum('cat_7_interest_num_active_days_to_overall_by_user').alias(
                'cat_7_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_7_num_requests_to_mean_by_category').alias('cat_7_num_requests_to_mean_by_category'),
            F.sum('cat_7_num_active_days_to_mean_by_category').alias('cat_7_num_active_days_to_mean_by_category'),
            F.sum('cat_8_interest_num_requests_to_overall_by_user').alias(
                'cat_8_interest_num_requests_to_overall_by_user'),
            F.sum('cat_8_interest_num_active_days_to_overall_by_user').alias(
                'cat_8_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_8_num_requests_to_mean_by_category').alias('cat_8_num_requests_to_mean_by_category'),
            F.sum('cat_8_num_active_days_to_mean_by_category').alias('cat_8_num_active_days_to_mean_by_category'),
            F.sum('cat_9_interest_num_requests_to_overall_by_user').alias(
                'cat_9_interest_num_requests_to_overall_by_user'),
            F.sum('cat_9_interest_num_active_days_to_overall_by_user').alias(
                'cat_9_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_9_num_requests_to_mean_by_category').alias('cat_9_num_requests_to_mean_by_category'),
            F.sum('cat_9_num_active_days_to_mean_by_category').alias('cat_9_num_active_days_to_mean_by_category'),
            F.sum('cat_10_interest_num_requests_to_overall_by_user').alias(
                'cat_10_interest_num_requests_to_overall_by_user'),
            F.sum('cat_10_interest_num_active_days_to_overall_by_user').alias(
                'cat_10_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_10_num_requests_to_mean_by_category').alias('cat_10_num_requests_to_mean_by_category'),
            F.sum('cat_10_num_active_days_to_mean_by_category').alias('cat_10_num_active_days_to_mean_by_category'),
            F.sum('cat_11_interest_num_requests_to_overall_by_user').alias(
                'cat_11_interest_num_requests_to_overall_by_user'),
            F.sum('cat_11_interest_num_active_days_to_overall_by_user').alias(
                'cat_11_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_11_num_requests_to_mean_by_category').alias('cat_11_num_requests_to_mean_by_category'),
            F.sum('cat_11_num_active_days_to_mean_by_category').alias('cat_11_num_active_days_to_mean_by_category'),
            F.sum('cat_12_interest_num_requests_to_overall_by_user').alias(
                'cat_12_interest_num_requests_to_overall_by_user'),
            F.sum('cat_12_interest_num_active_days_to_overall_by_user').alias(
                'cat_12_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_12_num_requests_to_mean_by_category').alias('cat_12_num_requests_to_mean_by_category'),
            F.sum('cat_12_num_active_days_to_mean_by_category').alias('cat_12_num_active_days_to_mean_by_category'),
            F.sum('cat_13_interest_num_requests_to_overall_by_user').alias(
                'cat_13_interest_num_requests_to_overall_by_user'),
            F.sum('cat_13_interest_num_active_days_to_overall_by_user').alias(
                'cat_13_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_13_num_requests_to_mean_by_category').alias('cat_13_num_requests_to_mean_by_category'),
            F.sum('cat_13_num_active_days_to_mean_by_category').alias('cat_13_num_active_days_to_mean_by_category'),
            F.sum('cat_14_interest_num_requests_to_overall_by_user').alias(
                'cat_14_interest_num_requests_to_overall_by_user'),
            F.sum('cat_14_interest_num_active_days_to_overall_by_user').alias(
                'cat_14_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_14_num_requests_to_mean_by_category').alias('cat_14_num_requests_to_mean_by_category'),
            F.sum('cat_14_num_active_days_to_mean_by_category').alias('cat_14_num_active_days_to_mean_by_category'),
            F.sum('cat_15_interest_num_requests_to_overall_by_user').alias(
                'cat_15_interest_num_requests_to_overall_by_user'),
            F.sum('cat_15_interest_num_active_days_to_overall_by_user').alias(
                'cat_15_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_15_num_requests_to_mean_by_category').alias('cat_15_num_requests_to_mean_by_category'),
            F.sum('cat_15_num_active_days_to_mean_by_category').alias('cat_15_num_active_days_to_mean_by_category'),
            F.sum('cat_16_interest_num_requests_to_overall_by_user').alias(
                'cat_16_interest_num_requests_to_overall_by_user'),
            F.sum('cat_16_interest_num_active_days_to_overall_by_user').alias(
                'cat_16_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_16_num_requests_to_mean_by_category').alias('cat_16_num_requests_to_mean_by_category'),
            F.sum('cat_16_num_active_days_to_mean_by_category').alias('cat_16_num_active_days_to_mean_by_category'),
            F.sum('cat_17_interest_num_requests_to_overall_by_user').alias(
                'cat_17_interest_num_requests_to_overall_by_user'),
            F.sum('cat_17_interest_num_active_days_to_overall_by_user').alias(
                'cat_17_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_17_num_requests_to_mean_by_category').alias('cat_17_num_requests_to_mean_by_category'),
            F.sum('cat_17_num_active_days_to_mean_by_category').alias('cat_17_num_active_days_to_mean_by_category'),
            F.sum('cat_18_interest_num_requests_to_overall_by_user').alias(
                'cat_18_interest_num_requests_to_overall_by_user'),
            F.sum('cat_18_interest_num_active_days_to_overall_by_user').alias(
                'cat_18_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_18_num_requests_to_mean_by_category').alias('cat_18_num_requests_to_mean_by_category'),
            F.sum('cat_18_num_active_days_to_mean_by_category').alias('cat_18_num_active_days_to_mean_by_category'),
            F.sum('cat_19_interest_num_requests_to_overall_by_user').alias(
                'cat_19_interest_num_requests_to_overall_by_user'),
            F.sum('cat_19_interest_num_active_days_to_overall_by_user').alias(
                'cat_19_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_19_num_requests_to_mean_by_category').alias('cat_19_num_requests_to_mean_by_category'),
            F.sum('cat_19_num_active_days_to_mean_by_category').alias('cat_19_num_active_days_to_mean_by_category'),
            F.sum('cat_20_interest_num_requests_to_overall_by_user').alias(
                'cat_20_interest_num_requests_to_overall_by_user'),
            F.sum('cat_20_interest_num_active_days_to_overall_by_user').alias(
                'cat_20_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_20_num_requests_to_mean_by_category').alias('cat_20_num_requests_to_mean_by_category'),
            F.sum('cat_20_num_active_days_to_mean_by_category').alias('cat_20_num_active_days_to_mean_by_category'),
            F.sum('cat_21_interest_num_requests_to_overall_by_user').alias(
                'cat_21_interest_num_requests_to_overall_by_user'),
            F.sum('cat_21_interest_num_active_days_to_overall_by_user').alias(
                'cat_21_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_21_num_requests_to_mean_by_category').alias('cat_21_num_requests_to_mean_by_category'),
            F.sum('cat_21_num_active_days_to_mean_by_category').alias('cat_21_num_active_days_to_mean_by_category'),
            F.sum('cat_22_interest_num_requests_to_overall_by_user').alias(
                'cat_22_interest_num_requests_to_overall_by_user'),
            F.sum('cat_22_interest_num_active_days_to_overall_by_user').alias(
                'cat_22_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_22_num_requests_to_mean_by_category').alias('cat_22_num_requests_to_mean_by_category'),
            F.sum('cat_22_num_active_days_to_mean_by_category').alias('cat_22_num_active_days_to_mean_by_category'),
            F.sum('cat_23_interest_num_requests_to_overall_by_user').alias(
                'cat_23_interest_num_requests_to_overall_by_user'),
            F.sum('cat_23_interest_num_active_days_to_overall_by_user').alias(
                'cat_23_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_23_num_requests_to_mean_by_category').alias('cat_23_num_requests_to_mean_by_category'),
            F.sum('cat_23_num_active_days_to_mean_by_category').alias('cat_23_num_active_days_to_mean_by_category'),
            F.sum('cat_24_interest_num_requests_to_overall_by_user').alias(
                'cat_24_interest_num_requests_to_overall_by_user'),
            F.sum('cat_24_interest_num_active_days_to_overall_by_user').alias(
                'cat_24_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_24_num_requests_to_mean_by_category').alias('cat_24_num_requests_to_mean_by_category'),
            F.sum('cat_24_num_active_days_to_mean_by_category').alias('cat_24_num_active_days_to_mean_by_category'),
            F.sum('cat_25_interest_num_requests_to_overall_by_user').alias(
                'cat_25_interest_num_requests_to_overall_by_user'),
            F.sum('cat_25_interest_num_active_days_to_overall_by_user').alias(
                'cat_25_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_25_num_requests_to_mean_by_category').alias('cat_25_num_requests_to_mean_by_category'),
            F.sum('cat_25_num_active_days_to_mean_by_category').alias('cat_25_num_active_days_to_mean_by_category'),
            F.sum('cat_26_interest_num_requests_to_overall_by_user').alias(
                'cat_26_interest_num_requests_to_overall_by_user'),
            F.sum('cat_26_interest_num_active_days_to_overall_by_user').alias(
                'cat_26_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_26_num_requests_to_mean_by_category').alias('cat_26_num_requests_to_mean_by_category'),
            F.sum('cat_26_num_active_days_to_mean_by_category').alias('cat_26_num_active_days_to_mean_by_category'),
            F.sum('cat_27_interest_num_requests_to_overall_by_user').alias(
                'cat_27_interest_num_requests_to_overall_by_user'),
            F.sum('cat_27_interest_num_active_days_to_overall_by_user').alias(
                'cat_27_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_27_num_requests_to_mean_by_category').alias('cat_27_num_requests_to_mean_by_category'),
            F.sum('cat_27_num_active_days_to_mean_by_category').alias('cat_27_num_active_days_to_mean_by_category'),
            F.sum('cat_28_interest_num_requests_to_overall_by_user').alias(
                'cat_28_interest_num_requests_to_overall_by_user'),
            F.sum('cat_28_interest_num_active_days_to_overall_by_user').alias(
                'cat_28_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_28_num_requests_to_mean_by_category').alias('cat_28_num_requests_to_mean_by_category'),
            F.sum('cat_28_num_active_days_to_mean_by_category').alias('cat_28_num_active_days_to_mean_by_category'),
            F.sum('cat_29_interest_num_requests_to_overall_by_user').alias(
                'cat_29_interest_num_requests_to_overall_by_user'),
            F.sum('cat_29_interest_num_active_days_to_overall_by_user').alias(
                'cat_29_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_29_num_requests_to_mean_by_category').alias('cat_29_num_requests_to_mean_by_category'),
            F.sum('cat_29_num_active_days_to_mean_by_category').alias('cat_29_num_active_days_to_mean_by_category'),
            F.sum('cat_30_interest_num_requests_to_overall_by_user').alias(
                'cat_30_interest_num_requests_to_overall_by_user'),
            F.sum('cat_30_interest_num_active_days_to_overall_by_user').alias(
                'cat_30_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_30_num_requests_to_mean_by_category').alias('cat_30_num_requests_to_mean_by_category'),
            F.sum('cat_30_num_active_days_to_mean_by_category').alias('cat_30_num_active_days_to_mean_by_category'),
            F.sum('cat_31_interest_num_requests_to_overall_by_user').alias(
                'cat_31_interest_num_requests_to_overall_by_user'),
            F.sum('cat_31_interest_num_active_days_to_overall_by_user').alias(
                'cat_31_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_31_num_requests_to_mean_by_category').alias('cat_31_num_requests_to_mean_by_category'),
            F.sum('cat_31_num_active_days_to_mean_by_category').alias('cat_31_num_active_days_to_mean_by_category'),
            F.sum('cat_32_interest_num_requests_to_overall_by_user').alias(
                'cat_32_interest_num_requests_to_overall_by_user'),
            F.sum('cat_32_interest_num_active_days_to_overall_by_user').alias(
                'cat_32_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_32_num_requests_to_mean_by_category').alias('cat_32_num_requests_to_mean_by_category'),
            F.sum('cat_32_num_active_days_to_mean_by_category').alias('cat_32_num_active_days_to_mean_by_category'),
            F.sum('cat_33_interest_num_requests_to_overall_by_user').alias(
                'cat_33_interest_num_requests_to_overall_by_user'),
            F.sum('cat_33_interest_num_active_days_to_overall_by_user').alias(
                'cat_33_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_33_num_requests_to_mean_by_category').alias('cat_33_num_requests_to_mean_by_category'),
            F.sum('cat_33_num_active_days_to_mean_by_category').alias('cat_33_num_active_days_to_mean_by_category'),
            F.sum('cat_34_interest_num_requests_to_overall_by_user').alias(
                'cat_34_interest_num_requests_to_overall_by_user'),
            F.sum('cat_34_interest_num_active_days_to_overall_by_user').alias(
                'cat_34_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_34_num_requests_to_mean_by_category').alias('cat_34_num_requests_to_mean_by_category'),
            F.sum('cat_34_num_active_days_to_mean_by_category').alias('cat_34_num_active_days_to_mean_by_category'),
            F.sum('cat_35_interest_num_requests_to_overall_by_user').alias(
                'cat_35_interest_num_requests_to_overall_by_user'),
            F.sum('cat_35_interest_num_active_days_to_overall_by_user').alias(
                'cat_35_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_35_num_requests_to_mean_by_category').alias('cat_35_num_requests_to_mean_by_category'),
            F.sum('cat_35_num_active_days_to_mean_by_category').alias('cat_35_num_active_days_to_mean_by_category'),
            F.sum('cat_36_interest_num_requests_to_overall_by_user').alias(
                'cat_36_interest_num_requests_to_overall_by_user'),
            F.sum('cat_36_interest_num_active_days_to_overall_by_user').alias(
                'cat_36_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_36_num_requests_to_mean_by_category').alias('cat_36_num_requests_to_mean_by_category'),
            F.sum('cat_36_num_active_days_to_mean_by_category').alias('cat_36_num_active_days_to_mean_by_category'),
            F.sum('cat_37_interest_num_requests_to_overall_by_user').alias(
                'cat_37_interest_num_requests_to_overall_by_user'),
            F.sum('cat_37_interest_num_active_days_to_overall_by_user').alias(
                'cat_37_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_37_num_requests_to_mean_by_category').alias('cat_37_num_requests_to_mean_by_category'),
            F.sum('cat_37_num_active_days_to_mean_by_category').alias('cat_37_num_active_days_to_mean_by_category'),
            F.sum('cat_38_interest_num_requests_to_overall_by_user').alias(
                'cat_38_interest_num_requests_to_overall_by_user'),
            F.sum('cat_38_interest_num_active_days_to_overall_by_user').alias(
                'cat_38_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_38_num_requests_to_mean_by_category').alias('cat_38_num_requests_to_mean_by_category'),
            F.sum('cat_38_num_active_days_to_mean_by_category').alias('cat_38_num_active_days_to_mean_by_category'),
            F.sum('cat_39_interest_num_requests_to_overall_by_user').alias(
                'cat_39_interest_num_requests_to_overall_by_user'),
            F.sum('cat_39_interest_num_active_days_to_overall_by_user').alias(
                'cat_39_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_39_num_requests_to_mean_by_category').alias('cat_39_num_requests_to_mean_by_category'),
            F.sum('cat_39_num_active_days_to_mean_by_category').alias('cat_39_num_active_days_to_mean_by_category'),
            F.sum('cat_40_interest_num_requests_to_overall_by_user').alias(
                'cat_40_interest_num_requests_to_overall_by_user'),
            F.sum('cat_40_interest_num_active_days_to_overall_by_user').alias(
                'cat_40_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_40_num_requests_to_mean_by_category').alias('cat_40_num_requests_to_mean_by_category'),
            F.sum('cat_40_num_active_days_to_mean_by_category').alias('cat_40_num_active_days_to_mean_by_category'),
            F.sum('cat_41_interest_num_requests_to_overall_by_user').alias(
                'cat_41_interest_num_requests_to_overall_by_user'),
            F.sum('cat_41_interest_num_active_days_to_overall_by_user').alias(
                'cat_41_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_41_num_requests_to_mean_by_category').alias('cat_41_num_requests_to_mean_by_category'),
            F.sum('cat_41_num_active_days_to_mean_by_category').alias('cat_41_num_active_days_to_mean_by_category'),
            F.sum('cat_42_interest_num_requests_to_overall_by_user').alias(
                'cat_42_interest_num_requests_to_overall_by_user'),
            F.sum('cat_42_interest_num_active_days_to_overall_by_user').alias(
                'cat_42_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_42_num_requests_to_mean_by_category').alias('cat_42_num_requests_to_mean_by_category'),
            F.sum('cat_42_num_active_days_to_mean_by_category').alias('cat_42_num_active_days_to_mean_by_category'),
            F.sum('cat_43_interest_num_requests_to_overall_by_user').alias(
                'cat_43_interest_num_requests_to_overall_by_user'),
            F.sum('cat_43_interest_num_active_days_to_overall_by_user').alias(
                'cat_43_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_43_num_requests_to_mean_by_category').alias('cat_43_num_requests_to_mean_by_category'),
            F.sum('cat_43_num_active_days_to_mean_by_category').alias('cat_43_num_active_days_to_mean_by_category'),
            F.sum('cat_44_interest_num_requests_to_overall_by_user').alias(
                'cat_44_interest_num_requests_to_overall_by_user'),
            F.sum('cat_44_interest_num_active_days_to_overall_by_user').alias(
                'cat_44_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_44_num_requests_to_mean_by_category').alias('cat_44_num_requests_to_mean_by_category'),
            F.sum('cat_44_num_active_days_to_mean_by_category').alias('cat_44_num_active_days_to_mean_by_category'),
            F.sum('cat_45_interest_num_requests_to_overall_by_user').alias(
                'cat_45_interest_num_requests_to_overall_by_user'),
            F.sum('cat_45_interest_num_active_days_to_overall_by_user').alias(
                'cat_45_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_45_num_requests_to_mean_by_category').alias('cat_45_num_requests_to_mean_by_category'),
            F.sum('cat_45_num_active_days_to_mean_by_category').alias('cat_45_num_active_days_to_mean_by_category'),
            F.sum('cat_46_interest_num_requests_to_overall_by_user').alias(
                'cat_46_interest_num_requests_to_overall_by_user'),
            F.sum('cat_46_interest_num_active_days_to_overall_by_user').alias(
                'cat_46_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_46_num_requests_to_mean_by_category').alias('cat_46_num_requests_to_mean_by_category'),
            F.sum('cat_46_num_active_days_to_mean_by_category').alias('cat_46_num_active_days_to_mean_by_category'),
            F.sum('cat_47_interest_num_requests_to_overall_by_user').alias(
                'cat_47_interest_num_requests_to_overall_by_user'),
            F.sum('cat_47_interest_num_active_days_to_overall_by_user').alias(
                'cat_47_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_47_num_requests_to_mean_by_category').alias('cat_47_num_requests_to_mean_by_category'),
            F.sum('cat_47_num_active_days_to_mean_by_category').alias('cat_47_num_active_days_to_mean_by_category'),
            F.sum('cat_48_interest_num_requests_to_overall_by_user').alias(
                'cat_48_interest_num_requests_to_overall_by_user'),
            F.sum('cat_48_interest_num_active_days_to_overall_by_user').alias(
                'cat_48_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_48_num_requests_to_mean_by_category').alias('cat_48_num_requests_to_mean_by_category'),
            F.sum('cat_48_num_active_days_to_mean_by_category').alias('cat_48_num_active_days_to_mean_by_category'),
            F.sum('cat_49_interest_num_requests_to_overall_by_user').alias(
                'cat_49_interest_num_requests_to_overall_by_user'),
            F.sum('cat_49_interest_num_active_days_to_overall_by_user').alias(
                'cat_49_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_49_num_requests_to_mean_by_category').alias('cat_49_num_requests_to_mean_by_category'),
            F.sum('cat_49_num_active_days_to_mean_by_category').alias('cat_49_num_active_days_to_mean_by_category'),
            F.sum('cat_50_interest_num_requests_to_overall_by_user').alias(
                'cat_50_interest_num_requests_to_overall_by_user'),
            F.sum('cat_50_interest_num_active_days_to_overall_by_user').alias(
                'cat_50_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_50_num_requests_to_mean_by_category').alias('cat_50_num_requests_to_mean_by_category'),
            F.sum('cat_50_num_active_days_to_mean_by_category').alias('cat_50_num_active_days_to_mean_by_category'),
            F.sum('cat_51_interest_num_requests_to_overall_by_user').alias(
                'cat_51_interest_num_requests_to_overall_by_user'),
            F.sum('cat_51_interest_num_active_days_to_overall_by_user').alias(
                'cat_51_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_51_num_requests_to_mean_by_category').alias('cat_51_num_requests_to_mean_by_category'),
            F.sum('cat_51_num_active_days_to_mean_by_category').alias('cat_51_num_active_days_to_mean_by_category'),
            F.sum('cat_52_interest_num_requests_to_overall_by_user').alias(
                'cat_52_interest_num_requests_to_overall_by_user'),
            F.sum('cat_52_interest_num_active_days_to_overall_by_user').alias(
                'cat_52_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_52_num_requests_to_mean_by_category').alias('cat_52_num_requests_to_mean_by_category'),
            F.sum('cat_52_num_active_days_to_mean_by_category').alias('cat_52_num_active_days_to_mean_by_category'),
            F.sum('cat_53_interest_num_requests_to_overall_by_user').alias(
                'cat_53_interest_num_requests_to_overall_by_user'),
            F.sum('cat_53_interest_num_active_days_to_overall_by_user').alias(
                'cat_53_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_53_num_requests_to_mean_by_category').alias('cat_53_num_requests_to_mean_by_category'),
            F.sum('cat_53_num_active_days_to_mean_by_category').alias('cat_53_num_active_days_to_mean_by_category'),
            F.sum('cat_54_interest_num_requests_to_overall_by_user').alias(
                'cat_54_interest_num_requests_to_overall_by_user'),
            F.sum('cat_54_interest_num_active_days_to_overall_by_user').alias(
                'cat_54_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_54_num_requests_to_mean_by_category').alias('cat_54_num_requests_to_mean_by_category'),
            F.sum('cat_54_num_active_days_to_mean_by_category').alias('cat_54_num_active_days_to_mean_by_category'),
            F.sum('cat_55_interest_num_requests_to_overall_by_user').alias(
                'cat_55_interest_num_requests_to_overall_by_user'),
            F.sum('cat_55_interest_num_active_days_to_overall_by_user').alias(
                'cat_55_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_55_num_requests_to_mean_by_category').alias('cat_55_num_requests_to_mean_by_category'),
            F.sum('cat_55_num_active_days_to_mean_by_category').alias('cat_55_num_active_days_to_mean_by_category'),
            F.sum('cat_56_interest_num_requests_to_overall_by_user').alias(
                'cat_56_interest_num_requests_to_overall_by_user'),
            F.sum('cat_56_interest_num_active_days_to_overall_by_user').alias(
                'cat_56_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_56_num_requests_to_mean_by_category').alias('cat_56_num_requests_to_mean_by_category'),
            F.sum('cat_56_num_active_days_to_mean_by_category').alias('cat_56_num_active_days_to_mean_by_category'),
            F.sum('cat_57_interest_num_requests_to_overall_by_user').alias(
                'cat_57_interest_num_requests_to_overall_by_user'),
            F.sum('cat_57_interest_num_active_days_to_overall_by_user').alias(
                'cat_57_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_57_num_requests_to_mean_by_category').alias('cat_57_num_requests_to_mean_by_category'),
            F.sum('cat_57_num_active_days_to_mean_by_category').alias('cat_57_num_active_days_to_mean_by_category'),
            F.sum('cat_58_interest_num_requests_to_overall_by_user').alias(
                'cat_58_interest_num_requests_to_overall_by_user'),
            F.sum('cat_58_interest_num_active_days_to_overall_by_user').alias(
                'cat_58_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_58_num_requests_to_mean_by_category').alias('cat_58_num_requests_to_mean_by_category'),
            F.sum('cat_58_num_active_days_to_mean_by_category').alias('cat_58_num_active_days_to_mean_by_category'),
            F.sum('cat_59_interest_num_requests_to_overall_by_user').alias(
                'cat_59_interest_num_requests_to_overall_by_user'),
            F.sum('cat_59_interest_num_active_days_to_overall_by_user').alias(
                'cat_59_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_59_num_requests_to_mean_by_category').alias('cat_59_num_requests_to_mean_by_category'),
            F.sum('cat_59_num_active_days_to_mean_by_category').alias('cat_59_num_active_days_to_mean_by_category'),
            F.sum('cat_60_interest_num_requests_to_overall_by_user').alias(
                'cat_60_interest_num_requests_to_overall_by_user'),
            F.sum('cat_60_interest_num_active_days_to_overall_by_user').alias(
                'cat_60_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_60_num_requests_to_mean_by_category').alias('cat_60_num_requests_to_mean_by_category'),
            F.sum('cat_60_num_active_days_to_mean_by_category').alias('cat_60_num_active_days_to_mean_by_category'),
            F.sum('cat_61_interest_num_requests_to_overall_by_user').alias(
                'cat_61_interest_num_requests_to_overall_by_user'),
            F.sum('cat_61_interest_num_active_days_to_overall_by_user').alias(
                'cat_61_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_61_num_requests_to_mean_by_category').alias('cat_61_num_requests_to_mean_by_category'),
            F.sum('cat_61_num_active_days_to_mean_by_category').alias('cat_61_num_active_days_to_mean_by_category'),
            F.sum('cat_62_interest_num_requests_to_overall_by_user').alias(
                'cat_62_interest_num_requests_to_overall_by_user'),
            F.sum('cat_62_interest_num_active_days_to_overall_by_user').alias(
                'cat_62_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_62_num_requests_to_mean_by_category').alias('cat_62_num_requests_to_mean_by_category'),
            F.sum('cat_62_num_active_days_to_mean_by_category').alias('cat_62_num_active_days_to_mean_by_category'),
            F.sum('cat_63_interest_num_requests_to_overall_by_user').alias(
                'cat_63_interest_num_requests_to_overall_by_user'),
            F.sum('cat_63_interest_num_active_days_to_overall_by_user').alias(
                'cat_63_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_63_num_requests_to_mean_by_category').alias('cat_63_num_requests_to_mean_by_category'),
            F.sum('cat_63_num_active_days_to_mean_by_category').alias('cat_63_num_active_days_to_mean_by_category'),
            F.sum('cat_64_interest_num_requests_to_overall_by_user').alias(
                'cat_64_interest_num_requests_to_overall_by_user'),
            F.sum('cat_64_interest_num_active_days_to_overall_by_user').alias(
                'cat_64_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_64_num_requests_to_mean_by_category').alias('cat_64_num_requests_to_mean_by_category'),
            F.sum('cat_64_num_active_days_to_mean_by_category').alias('cat_64_num_active_days_to_mean_by_category'),
            F.sum('cat_65_interest_num_requests_to_overall_by_user').alias(
                'cat_65_interest_num_requests_to_overall_by_user'),
            F.sum('cat_65_interest_num_active_days_to_overall_by_user').alias(
                'cat_65_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_65_num_requests_to_mean_by_category').alias('cat_65_num_requests_to_mean_by_category'),
            F.sum('cat_65_num_active_days_to_mean_by_category').alias('cat_65_num_active_days_to_mean_by_category'),
            F.sum('cat_66_interest_num_requests_to_overall_by_user').alias(
                'cat_66_interest_num_requests_to_overall_by_user'),
            F.sum('cat_66_interest_num_active_days_to_overall_by_user').alias(
                'cat_66_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_66_num_requests_to_mean_by_category').alias('cat_66_num_requests_to_mean_by_category'),
            F.sum('cat_66_num_active_days_to_mean_by_category').alias('cat_66_num_active_days_to_mean_by_category'),
            F.sum('cat_67_interest_num_requests_to_overall_by_user').alias(
                'cat_67_interest_num_requests_to_overall_by_user'),
            F.sum('cat_67_interest_num_active_days_to_overall_by_user').alias(
                'cat_67_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_67_num_requests_to_mean_by_category').alias('cat_67_num_requests_to_mean_by_category'),
            F.sum('cat_67_num_active_days_to_mean_by_category').alias('cat_67_num_active_days_to_mean_by_category'),
            F.sum('cat_68_interest_num_requests_to_overall_by_user').alias(
                'cat_68_interest_num_requests_to_overall_by_user'),
            F.sum('cat_68_interest_num_active_days_to_overall_by_user').alias(
                'cat_68_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_68_num_requests_to_mean_by_category').alias('cat_68_num_requests_to_mean_by_category'),
            F.sum('cat_68_num_active_days_to_mean_by_category').alias('cat_68_num_active_days_to_mean_by_category'),
            F.sum('cat_69_interest_num_requests_to_overall_by_user').alias(
                'cat_69_interest_num_requests_to_overall_by_user'),
            F.sum('cat_69_interest_num_active_days_to_overall_by_user').alias(
                'cat_69_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_69_num_requests_to_mean_by_category').alias('cat_69_num_requests_to_mean_by_category'),
            F.sum('cat_69_num_active_days_to_mean_by_category').alias('cat_69_num_active_days_to_mean_by_category'),
            F.sum('cat_70_interest_num_requests_to_overall_by_user').alias(
                'cat_70_interest_num_requests_to_overall_by_user'),
            F.sum('cat_70_interest_num_active_days_to_overall_by_user').alias(
                'cat_70_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_70_num_requests_to_mean_by_category').alias('cat_70_num_requests_to_mean_by_category'),
            F.sum('cat_70_num_active_days_to_mean_by_category').alias('cat_70_num_active_days_to_mean_by_category'),
            F.sum('cat_71_interest_num_requests_to_overall_by_user').alias(
                'cat_71_interest_num_requests_to_overall_by_user'),
            F.sum('cat_71_interest_num_active_days_to_overall_by_user').alias(
                'cat_71_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_71_num_requests_to_mean_by_category').alias('cat_71_num_requests_to_mean_by_category'),
            F.sum('cat_71_num_active_days_to_mean_by_category').alias('cat_71_num_active_days_to_mean_by_category'),
            F.sum('cat_72_interest_num_requests_to_overall_by_user').alias(
                'cat_72_interest_num_requests_to_overall_by_user'),
            F.sum('cat_72_interest_num_active_days_to_overall_by_user').alias(
                'cat_72_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_72_num_requests_to_mean_by_category').alias('cat_72_num_requests_to_mean_by_category'),
            F.sum('cat_72_num_active_days_to_mean_by_category').alias('cat_72_num_active_days_to_mean_by_category'),
            F.sum('cat_73_interest_num_requests_to_overall_by_user').alias(
                'cat_73_interest_num_requests_to_overall_by_user'),
            F.sum('cat_73_interest_num_active_days_to_overall_by_user').alias(
                'cat_73_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_73_num_requests_to_mean_by_category').alias('cat_73_num_requests_to_mean_by_category'),
            F.sum('cat_73_num_active_days_to_mean_by_category').alias('cat_73_num_active_days_to_mean_by_category'),
            F.sum('cat_74_interest_num_requests_to_overall_by_user').alias(
                'cat_74_interest_num_requests_to_overall_by_user'),
            F.sum('cat_74_interest_num_active_days_to_overall_by_user').alias(
                'cat_74_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_74_num_requests_to_mean_by_category').alias('cat_74_num_requests_to_mean_by_category'),
            F.sum('cat_74_num_active_days_to_mean_by_category').alias('cat_74_num_active_days_to_mean_by_category'),
            F.sum('cat_75_interest_num_requests_to_overall_by_user').alias(
                'cat_75_interest_num_requests_to_overall_by_user'),
            F.sum('cat_75_interest_num_active_days_to_overall_by_user').alias(
                'cat_75_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_75_num_requests_to_mean_by_category').alias('cat_75_num_requests_to_mean_by_category'),
            F.sum('cat_75_num_active_days_to_mean_by_category').alias('cat_75_num_active_days_to_mean_by_category'),
            F.sum('cat_76_interest_num_requests_to_overall_by_user').alias(
                'cat_76_interest_num_requests_to_overall_by_user'),
            F.sum('cat_76_interest_num_active_days_to_overall_by_user').alias(
                'cat_76_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_76_num_requests_to_mean_by_category').alias('cat_76_num_requests_to_mean_by_category'),
            F.sum('cat_76_num_active_days_to_mean_by_category').alias('cat_76_num_active_days_to_mean_by_category'),
            F.sum('cat_77_interest_num_requests_to_overall_by_user').alias(
                'cat_77_interest_num_requests_to_overall_by_user'),
            F.sum('cat_77_interest_num_active_days_to_overall_by_user').alias(
                'cat_77_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_77_num_requests_to_mean_by_category').alias('cat_77_num_requests_to_mean_by_category'),
            F.sum('cat_77_num_active_days_to_mean_by_category').alias('cat_77_num_active_days_to_mean_by_category'),
            F.sum('cat_78_interest_num_requests_to_overall_by_user').alias(
                'cat_78_interest_num_requests_to_overall_by_user'),
            F.sum('cat_78_interest_num_active_days_to_overall_by_user').alias(
                'cat_78_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_78_num_requests_to_mean_by_category').alias('cat_78_num_requests_to_mean_by_category'),
            F.sum('cat_78_num_active_days_to_mean_by_category').alias('cat_78_num_active_days_to_mean_by_category'),
            F.sum('cat_79_interest_num_requests_to_overall_by_user').alias(
                'cat_79_interest_num_requests_to_overall_by_user'),
            F.sum('cat_79_interest_num_active_days_to_overall_by_user').alias(
                'cat_79_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_79_num_requests_to_mean_by_category').alias('cat_79_num_requests_to_mean_by_category'),
            F.sum('cat_79_num_active_days_to_mean_by_category').alias('cat_79_num_active_days_to_mean_by_category'),
            F.sum('cat_80_interest_num_requests_to_overall_by_user').alias(
                'cat_80_interest_num_requests_to_overall_by_user'),
            F.sum('cat_80_interest_num_active_days_to_overall_by_user').alias(
                'cat_80_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_80_num_requests_to_mean_by_category').alias('cat_80_num_requests_to_mean_by_category'),
            F.sum('cat_80_num_active_days_to_mean_by_category').alias('cat_80_num_active_days_to_mean_by_category'),
            F.sum('cat_81_interest_num_requests_to_overall_by_user').alias(
                'cat_81_interest_num_requests_to_overall_by_user'),
            F.sum('cat_81_interest_num_active_days_to_overall_by_user').alias(
                'cat_81_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_81_num_requests_to_mean_by_category').alias('cat_81_num_requests_to_mean_by_category'),
            F.sum('cat_81_num_active_days_to_mean_by_category').alias('cat_81_num_active_days_to_mean_by_category'),
            F.sum('cat_82_interest_num_requests_to_overall_by_user').alias(
                'cat_82_interest_num_requests_to_overall_by_user'),
            F.sum('cat_82_interest_num_active_days_to_overall_by_user').alias(
                'cat_82_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_82_num_requests_to_mean_by_category').alias('cat_82_num_requests_to_mean_by_category'),
            F.sum('cat_82_num_active_days_to_mean_by_category').alias('cat_82_num_active_days_to_mean_by_category'),
            F.sum('cat_83_interest_num_requests_to_overall_by_user').alias(
                'cat_83_interest_num_requests_to_overall_by_user'),
            F.sum('cat_83_interest_num_active_days_to_overall_by_user').alias(
                'cat_83_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_83_num_requests_to_mean_by_category').alias('cat_83_num_requests_to_mean_by_category'),
            F.sum('cat_83_num_active_days_to_mean_by_category').alias('cat_83_num_active_days_to_mean_by_category'),
            F.sum('cat_84_interest_num_requests_to_overall_by_user').alias(
                'cat_84_interest_num_requests_to_overall_by_user'),
            F.sum('cat_84_interest_num_active_days_to_overall_by_user').alias(
                'cat_84_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_84_num_requests_to_mean_by_category').alias('cat_84_num_requests_to_mean_by_category'),
            F.sum('cat_84_num_active_days_to_mean_by_category').alias('cat_84_num_active_days_to_mean_by_category'),
            F.sum('cat_85_interest_num_requests_to_overall_by_user').alias(
                'cat_85_interest_num_requests_to_overall_by_user'),
            F.sum('cat_85_interest_num_active_days_to_overall_by_user').alias(
                'cat_85_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_85_num_requests_to_mean_by_category').alias('cat_85_num_requests_to_mean_by_category'),
            F.sum('cat_85_num_active_days_to_mean_by_category').alias('cat_85_num_active_days_to_mean_by_category'),
            F.sum('cat_86_interest_num_requests_to_overall_by_user').alias(
                'cat_86_interest_num_requests_to_overall_by_user'),
            F.sum('cat_86_interest_num_active_days_to_overall_by_user').alias(
                'cat_86_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_86_num_requests_to_mean_by_category').alias('cat_86_num_requests_to_mean_by_category'),
            F.sum('cat_86_num_active_days_to_mean_by_category').alias('cat_86_num_active_days_to_mean_by_category'),
            F.sum('cat_87_interest_num_requests_to_overall_by_user').alias(
                'cat_87_interest_num_requests_to_overall_by_user'),
            F.sum('cat_87_interest_num_active_days_to_overall_by_user').alias(
                'cat_87_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_87_num_requests_to_mean_by_category').alias('cat_87_num_requests_to_mean_by_category'),
            F.sum('cat_87_num_active_days_to_mean_by_category').alias('cat_87_num_active_days_to_mean_by_category'),
            F.sum('cat_88_interest_num_requests_to_overall_by_user').alias(
                'cat_88_interest_num_requests_to_overall_by_user'),
            F.sum('cat_88_interest_num_active_days_to_overall_by_user').alias(
                'cat_88_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_88_num_requests_to_mean_by_category').alias('cat_88_num_requests_to_mean_by_category'),
            F.sum('cat_88_num_active_days_to_mean_by_category').alias('cat_88_num_active_days_to_mean_by_category'),
            F.sum('cat_89_interest_num_requests_to_overall_by_user').alias(
                'cat_89_interest_num_requests_to_overall_by_user'),
            F.sum('cat_89_interest_num_active_days_to_overall_by_user').alias(
                'cat_89_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_89_num_requests_to_mean_by_category').alias('cat_89_num_requests_to_mean_by_category'),
            F.sum('cat_89_num_active_days_to_mean_by_category').alias('cat_89_num_active_days_to_mean_by_category'),
            F.sum('cat_90_interest_num_requests_to_overall_by_user').alias(
                'cat_90_interest_num_requests_to_overall_by_user'),
            F.sum('cat_90_interest_num_active_days_to_overall_by_user').alias(
                'cat_90_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_90_num_requests_to_mean_by_category').alias('cat_90_num_requests_to_mean_by_category'),
            F.sum('cat_90_num_active_days_to_mean_by_category').alias('cat_90_num_active_days_to_mean_by_category'),
            F.sum('cat_91_interest_num_requests_to_overall_by_user').alias(
                'cat_91_interest_num_requests_to_overall_by_user'),
            F.sum('cat_91_interest_num_active_days_to_overall_by_user').alias(
                'cat_91_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_91_num_requests_to_mean_by_category').alias('cat_91_num_requests_to_mean_by_category'),
            F.sum('cat_91_num_active_days_to_mean_by_category').alias('cat_91_num_active_days_to_mean_by_category'),
            F.sum('cat_92_interest_num_requests_to_overall_by_user').alias(
                'cat_92_interest_num_requests_to_overall_by_user'),
            F.sum('cat_92_interest_num_active_days_to_overall_by_user').alias(
                'cat_92_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_92_num_requests_to_mean_by_category').alias('cat_92_num_requests_to_mean_by_category'),
            F.sum('cat_92_num_active_days_to_mean_by_category').alias('cat_92_num_active_days_to_mean_by_category'),
            F.sum('cat_93_interest_num_requests_to_overall_by_user').alias(
                'cat_93_interest_num_requests_to_overall_by_user'),
            F.sum('cat_93_interest_num_active_days_to_overall_by_user').alias(
                'cat_93_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_93_num_requests_to_mean_by_category').alias('cat_93_num_requests_to_mean_by_category'),
            F.sum('cat_93_num_active_days_to_mean_by_category').alias('cat_93_num_active_days_to_mean_by_category'),
            F.sum('cat_94_interest_num_requests_to_overall_by_user').alias(
                'cat_94_interest_num_requests_to_overall_by_user'),
            F.sum('cat_94_interest_num_active_days_to_overall_by_user').alias(
                'cat_94_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_94_num_requests_to_mean_by_category').alias('cat_94_num_requests_to_mean_by_category'),
            F.sum('cat_94_num_active_days_to_mean_by_category').alias('cat_94_num_active_days_to_mean_by_category'),
            F.sum('cat_95_interest_num_requests_to_overall_by_user').alias(
                'cat_95_interest_num_requests_to_overall_by_user'),
            F.sum('cat_95_interest_num_active_days_to_overall_by_user').alias(
                'cat_95_interest_num_active_days_to_overall_by_user'),
            F.sum('cat_95_num_requests_to_mean_by_category').alias('cat_95_num_requests_to_mean_by_category'),
            F.sum('cat_95_num_active_days_to_mean_by_category').alias('cat_95_num_active_days_to_mean_by_category')
        )
    )

    return final_df


In [5]:
log = logging.getLogger(__name__)
logical_date = datetime.now().date()

In [6]:
build_host_interests(spark, logical_date, log)

DataFrame[regid: double, app_n: double, msisdn: int, cat_1_interest_num_requests_to_overall_by_user: double, cat_1_interest_num_active_days_to_overall_by_user: double, cat_1_num_requests_to_mean_by_category: double, cat_1_num_active_days_to_mean_by_category: double, cat_2_interest_num_requests_to_overall_by_user: double, cat_2_interest_num_active_days_to_overall_by_user: double, cat_2_num_requests_to_mean_by_category: double, cat_2_num_active_days_to_mean_by_category: double, cat_3_interest_num_requests_to_overall_by_user: double, cat_3_interest_num_active_days_to_overall_by_user: double, cat_3_num_requests_to_mean_by_category: double, cat_3_num_active_days_to_mean_by_category: double, cat_4_interest_num_requests_to_overall_by_user: double, cat_4_interest_num_active_days_to_overall_by_user: double, cat_4_num_requests_to_mean_by_category: double, cat_4_num_active_days_to_mean_by_category: double, cat_5_interest_num_requests_to_overall_by_user: double, cat_5_interest_num_active_days_to_o